# 📐 Notebook 03 — Triplet Generation for Bi-Encoder Training

**CV Matcher Pro** | Capstone Project

This notebook creates training triplets from our cleaned JD and resume data:
- **Anchor** = real job description text
- **Positive** = resume chunk from the SAME domain
- **Negative** = resume chunk from a DIFFERENT domain

We also apply:
1. Hard negative mining (similar-domain negatives)
2. Back-translation to Indonesian for bilingual augmentation
3. Stratified train/val/test splitting with leakage checks

---
## Section 1 · Setup & Data Loading

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import random
import os
from collections import Counter

# Paths
BASE_DIR = '/content/drive/MyDrive/CVPRO'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
TRIPLET_DIR = os.path.join(BASE_DIR, 'data', 'triplets')

os.makedirs(TRIPLET_DIR, exist_ok=True)

print("📂 Loading cleaned datasets...")
jd_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'jd_clean.csv'))
resume_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'resume_clean.csv'))

print(f"   JD rows      : {len(jd_df):,}")
print(f"   Resume rows   : {len(resume_df):,}")
print(f"   JD domains    : {sorted(jd_df['domain'].unique())}")
print(f"   Resume domains: {sorted(resume_df['domain'].unique())}")

📂 Loading cleaned datasets...
   JD rows      : 285,206
   Resume rows   : 21,480
   JD domains    : ['Education', 'Finance', 'HR', 'Healthcare', 'IT', 'Legal', 'Marketing', 'Operational', 'Other', 'Sales']
   Resume domains: ['Education', 'Finance', 'HR', 'Healthcare', 'IT', 'Legal', 'Marketing', 'Operational', 'Other', 'Sales']


In [3]:
# Quick sanity check — expected columns
assert 'jd_text' in jd_df.columns, "Missing 'jd_text' column in jd_clean.csv"
assert 'domain' in jd_df.columns, "Missing 'domain' column in jd_clean.csv"
assert 'resume_chunk' in resume_df.columns, "Missing 'resume_chunk' column in resume_clean.csv"
assert 'domain' in resume_df.columns, "Missing 'domain' column in resume_clean.csv"

print("✅ Column checks passed")

# Show domain distribution
print("\n📊 JD domain counts:")
print(jd_df['domain'].value_counts().to_string())
print("\n📊 Resume domain counts:")
print(resume_df['domain'].value_counts().to_string())

✅ Column checks passed

📊 JD domain counts:
domain
Healthcare     97932
Operational    54281
IT             29188
Other          24093
Finance        22552
HR             15017
Education      14899
Sales          12019
Legal           9376
Marketing       5849

📊 Resume domain counts:
domain
Operational    4374
Finance        3428
IT             2602
Other          2589
Education      2540
Healthcare     2239
Sales          1252
Marketing       999
Legal           937
HR              520


In [4]:
# ============================================================================
# Filter out 'Other' domain — it's a noisy mix of unclassifiable JDs/resumes
# that would hurt triplet quality
# ============================================================================
EXCLUDE_DOMAINS = ['Other']

before_jd = len(jd_df)
before_res = len(resume_df)
jd_df = jd_df[~jd_df['domain'].isin(EXCLUDE_DOMAINS)].reset_index(drop=True)
resume_df = resume_df[~resume_df['domain'].isin(EXCLUDE_DOMAINS)].reset_index(drop=True)

print(f"\n🧹 Excluded domains {EXCLUDE_DOMAINS}:")
print(f"   JD:     {before_jd:,} → {len(jd_df):,} (removed {before_jd - len(jd_df):,})")
print(f"   Resume: {before_res:,} → {len(resume_df):,} (removed {before_res - len(resume_df):,})")
print(f"   Remaining domains: {sorted(jd_df['domain'].unique())}")


🧹 Excluded domains ['Other']:
   JD:     285,206 → 261,113 (removed 24,093)
   Resume: 21,480 → 18,891 (removed 2,589)
   Remaining domains: ['Education', 'Finance', 'HR', 'Healthcare', 'IT', 'Legal', 'Marketing', 'Operational', 'Sales']


---
## Section 2 · Triplet Generation Strategy

### How we build each triplet:

| Component    | Source            | Rule                                    |
|:-------------|:------------------|:----------------------------------------|
| **Anchor**   | `jd_clean.csv`    | Real job description text               |
| **Positive** | `resume_clean.csv`| Resume chunk from the **same** domain   |
| **Negative** | `resume_clean.csv`| Resume chunk from a **different** domain|

### Why this works:
- Both JD and resume texts are **real-world documents** (not synthetic)
- Domain alignment ensures semantic relevance (IT JD ↔ IT resume)
- The model learns: *"this resume matches this job"* vs *"this resume does NOT match"*

### Scale target:
- ~1,000 triplets per domain → ~7,000 total (before augmentation)
- After Indonesian back-translation → ~9,000+ triplets

---
## Section 3 · Generate Base Triplets

In [5]:
def generate_real_triplets(jd_df, resume_df, n_per_domain=1000, seed=42):
    """
    Generate (anchor, positive, negative) triplets from real JD and resume data.

    Strategy:
        - anchor   = random JD from domain D
        - positive = random resume from domain D  (SAME domain)
        - negative = random resume from domain ≠D (DIFFERENT domain)

    Args:
        jd_df:          DataFrame with columns ['jd_text', 'domain']
        resume_df:      DataFrame with columns ['resume_chunk', 'domain']
        n_per_domain:   Max triplets to generate per domain
        seed:           Random seed for reproducibility

    Returns:
        DataFrame with columns ['anchor', 'positive', 'negative', 'domain']
    """
    random.seed(seed)
    np.random.seed(seed)
    triplets = []

    # Only use domains present in BOTH datasets
    common_domains = sorted(
        set(jd_df['domain'].unique()) & set(resume_df['domain'].unique())
    )
    print(f"🔗 Common domains ({len(common_domains)}): {common_domains}")

    for domain in common_domains:
        # Build pools
        jd_pool = jd_df[jd_df['domain'] == domain]['jd_text'].dropna().tolist()
        pos_pool = resume_df[resume_df['domain'] == domain]['resume_chunk'].dropna().tolist()

        neg_domains = [d for d in common_domains if d != domain]
        neg_pool = resume_df[resume_df['domain'].isin(neg_domains)]['resume_chunk'].dropna().tolist()

        if not jd_pool or not pos_pool or not neg_pool:
            print(f"   ⚠️  Skipping '{domain}' — insufficient data "
                  f"(JD={len(jd_pool)}, pos={len(pos_pool)}, neg={len(neg_pool)})")
            continue

        # Cap to avoid excessive re-sampling from tiny pools
        # Allow up to 3× pool size to keep diversity reasonable
        actual_n = min(n_per_domain, len(jd_pool) * 3, len(pos_pool) * 3)

        for _ in range(actual_n):
            triplets.append({
                'anchor': random.choice(jd_pool),
                'positive': random.choice(pos_pool),
                'negative': random.choice(neg_pool),
                'domain': domain,
            })

        print(f"   ✅ {domain:<15s} → {actual_n:,} triplets  "
              f"(JD pool={len(jd_pool):,}, pos pool={len(pos_pool):,}, neg pool={len(neg_pool):,})")

    df = pd.DataFrame(triplets)
    print(f"\n📐 Total base triplets: {len(df):,}")
    return df

In [6]:
triplet_df = generate_real_triplets(jd_df, resume_df, n_per_domain=1000, seed=42)
triplet_df.head()

🔗 Common domains (9): ['Education', 'Finance', 'HR', 'Healthcare', 'IT', 'Legal', 'Marketing', 'Operational', 'Sales']
   ✅ Education       → 1,000 triplets  (JD pool=14,899, pos pool=2,540, neg pool=16,351)
   ✅ Finance         → 1,000 triplets  (JD pool=22,552, pos pool=3,428, neg pool=15,463)
   ✅ HR              → 1,000 triplets  (JD pool=15,017, pos pool=520, neg pool=18,371)
   ✅ Healthcare      → 1,000 triplets  (JD pool=97,932, pos pool=2,239, neg pool=16,652)
   ✅ IT              → 1,000 triplets  (JD pool=29,188, pos pool=2,602, neg pool=16,289)
   ✅ Legal           → 1,000 triplets  (JD pool=9,376, pos pool=937, neg pool=17,954)
   ✅ Marketing       → 1,000 triplets  (JD pool=5,849, pos pool=999, neg pool=17,892)
   ✅ Operational     → 1,000 triplets  (JD pool=54,281, pos pool=4,374, neg pool=14,517)
   ✅ Sales           → 1,000 triplets  (JD pool=12,019, pos pool=1,252, neg pool=17,639)

📐 Total base triplets: 9,000


,anchor,positive,negative,domain
0,Note: If you are CURRENTLY employed at Childre...,education setting. My duties included 1on1 stu...,TERRITORY HR MANAGER Executive Profile Territo...,Education
1,Title: Bowling Volunteer Coach Reports to: Ath...,Summary To be employed as an Administrative As...,ADMINISTRATOR Executive Profile Accomplished E...,Education
2,Job Description Golden Nugget Atlantic City ha...,projects. Created hands on activities for stud...,Education Master of Professional Accountancy :...,Education
3,Cover Supervisor/Cover Teacher - Mole Valley E...,skills to help with personal growth in additio...,BUSINESS SYSTEMS ANALYST I Qualifications TECH...,Education
4,To complete credit investigation for received ...,Education Master of Science : Technology Comme...,"Skills, Outlook, Powerpoint, Proofreading, Rec...",Education


---
## Section 4 · Hard Negative Mining (Semi-Hard Negatives)

Standard negatives come from any *different* domain, but **semi-hard negatives**
come from a *similar* domain — making the model work harder to discriminate.

| Domain       | Similar / Confusable Domains         |
|:-------------|:-------------------------------------|
| IT           | Engineering, BusinessDev             |
| Finance      | BusinessDev                          |
| HR           | BusinessDev, Sales                   |
| Sales        | Marketing, BusinessDev               |
| BusinessDev  | Sales, IT, Finance                   |
| Healthcare   | *(none — quite distinct)*            |
| Marketing    | Sales, BusinessDev                   |

In [7]:
# Domain similarity map — each domain maps to its "confusable" neighbors
SIMILAR_DOMAINS = {
    'IT':          ['BusinessDev'],
    'Finance':     ['BusinessDev'],
    'HR':          ['BusinessDev', 'Sales'],
    'Sales':       ['Marketing', 'BusinessDev'],
    'BusinessDev': ['Sales', 'IT', 'Finance'],
    'Healthcare':  [],                          # distinct enough
    'Marketing':   ['Sales', 'BusinessDev'],
}


def generate_hard_negatives(jd_df, resume_df, similar_map, n_per_domain=200, seed=123):
    """
    Generate triplets where the negative comes from a SIMILAR (confusable) domain
    instead of a random different domain.  These 'semi-hard' negatives force the
    model to learn finer-grained distinctions.

    Returns:
        DataFrame with columns ['anchor', 'positive', 'negative', 'domain', 'neg_type']
    """
    random.seed(seed)
    triplets = []

    common_domains = sorted(
        set(jd_df['domain'].unique()) & set(resume_df['domain'].unique())
    )

    for domain in common_domains:
        # Find which similar domains actually exist in our data
        sim_domains = [d for d in similar_map.get(domain, []) if d in common_domains]
        if not sim_domains:
            continue  # no confusable neighbors available

        jd_pool = jd_df[jd_df['domain'] == domain]['jd_text'].dropna().tolist()
        pos_pool = resume_df[resume_df['domain'] == domain]['resume_chunk'].dropna().tolist()
        hard_neg_pool = resume_df[resume_df['domain'].isin(sim_domains)]['resume_chunk'].dropna().tolist()

        if not jd_pool or not pos_pool or not hard_neg_pool:
            continue

        actual_n = min(n_per_domain, len(jd_pool) * 2, len(pos_pool) * 2)

        for _ in range(actual_n):
            triplets.append({
                'anchor': random.choice(jd_pool),
                'positive': random.choice(pos_pool),
                'negative': random.choice(hard_neg_pool),
                'domain': domain,
                'neg_type': 'hard',
            })

        print(f"   🔶 {domain:<15s} → {actual_n:,} hard-negative triplets  "
              f"(sim domains: {sim_domains})")

    df = pd.DataFrame(triplets)
    print(f"\n🔶 Total hard-negative triplets: {len(df):,}")
    return df

In [8]:
hard_df = generate_hard_negatives(jd_df, resume_df, SIMILAR_DOMAINS, n_per_domain=200, seed=123)

# Tag the base triplets as 'random' negatives
triplet_df['neg_type'] = 'random'

# Combine base + hard negatives
all_triplets = pd.concat([triplet_df, hard_df], ignore_index=True)
print(f"\n📐 Combined triplets: {len(all_triplets):,}")
print(all_triplets['neg_type'].value_counts())

   🔶 HR              → 200 hard-negative triplets  (sim domains: ['Sales'])
   🔶 Marketing       → 200 hard-negative triplets  (sim domains: ['Sales'])
   🔶 Sales           → 200 hard-negative triplets  (sim domains: ['Marketing'])

🔶 Total hard-negative triplets: 600

📐 Combined triplets: 9,600
neg_type
random    9000
hard       600
Name: count, dtype: int64


---
## Section 5 · Data Augmentation — Back-Translation to Indonesian

We translate a subset of triplets (anchor + positive) from English → Indonesian
to create bilingual training pairs.  This helps the model handle Indonesian CVs/JDs
at inference time.

We use `googletrans==4.0.0-rc1` (free, no API key needed).
Only **anchor** and **positive** are translated — the negative stays in English
to maintain clear contrast.

In [9]:
# ============================================================================
# Section 5 · Indonesian Back-Translation (using deep-translator)
# ============================================================================
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.5 MB/s eta 0:00:00


In [10]:
from deep_translator import GoogleTranslator
import time
translator = GoogleTranslator(source='en', target='id')
def translate_to_id(text, max_chars=4500):
    """Translate English text to Indonesian. Handles length limits."""
    try:
        text = str(text)[:max_chars]
        result = translator.translate(text)
        return result if result else text
    except Exception as e:
        return text
# Augment 30% of triplets with Indonesian translation
AUGMENT_RATIO = 0.30
n_augment = int(len(all_triplets) * AUGMENT_RATIO)
print(f"🌐 Translating {n_augment} triplets to Indonesian...")
print(f"   (This will take ~10-20 minutes)")
aug_indices = all_triplets.sample(n=n_augment, random_state=42).index
aug_rows = []
for i, idx in enumerate(aug_indices):
    row = all_triplets.loc[idx]

    anchor_id = translate_to_id(row['anchor'])
    positive_id = translate_to_id(row['positive'])
    # Negative stays English (untuk kontras)

    aug_rows.append({
        'anchor': anchor_id,
        'positive': positive_id,
        'negative': row['negative'],
        'domain': row['domain'],
        'lang': 'id',
        'neg_type': row.get('neg_type', 'translated'),
    })

    if (i + 1) % 100 == 0:
        print(f"   Translated {i+1}/{n_augment} ({(i+1)/n_augment*100:.0f}%)")
        time.sleep(1)  # Rate limit protection
aug_df = pd.DataFrame(aug_rows)
print(f"\n✅ Indonesian augmentation: {len(aug_df):,} triplets")
# Combine
all_triplets_aug = pd.concat([all_triplets, aug_df], ignore_index=True)
print(f"📐 Total triplets (with augmentation): {len(all_triplets_aug):,}")
print(f"   Per domain:\n{all_triplets_aug['domain'].value_counts().to_string()}")

🌐 Translating 2880 triplets to Indonesian...
   (This will take ~10-20 minutes)
   Translated 100/2880 (3%)
   Translated 200/2880 (7%)
   Translated 300/2880 (10%)
   Translated 400/2880 (14%)
   Translated 500/2880 (17%)
   Translated 600/2880 (21%)
   Translated 700/2880 (24%)
   Translated 800/2880 (28%)
   Translated 900/2880 (31%)
   Translated 1000/2880 (35%)
   Translated 1100/2880 (38%)
   Translated 1200/2880 (42%)
   Translated 1300/2880 (45%)
   Translated 1400/2880 (49%)
   Translated 1500/2880 (52%)
   Translated 1600/2880 (56%)
   Translated 1700/2880 (59%)
   Translated 1800/2880 (62%)
   Translated 1900/2880 (66%)
   Translated 2000/2880 (69%)
   Translated 2100/2880 (73%)
   Translated 2200/2880 (76%)
   Translated 2300/2880 (80%)
   Translated 2400/2880 (83%)
   Translated 2500/2880 (87%)
   Translated 2600/2880 (90%)
   Translated 2700/2880 (94%)
   Translated 2800/2880 (97%)

✅ Indonesian augmentation: 2,880 triplets
📐 Total triplets (with augmentation): 12,480
   

In [11]:
full_triplets = all_triplets_aug

---
## Section 6 · Train / Val / Test Split

- **80%** train · **10%** validation · **10%** test
- Stratified by domain to maintain balance across splits
- Data leakage check: ensure no resume chunk appears in both train and test

In [12]:
from sklearn.model_selection import train_test_split

def stratified_split(df, train_ratio=0.80, val_ratio=0.10, test_ratio=0.10, seed=42):
    """
    Split triplets into train/val/test with stratification by domain.

    Returns:
        (train_df, val_df, test_df)
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Ratios must sum to 1.0"

    # First split: train vs (val+test)
    train_df, temp_df = train_test_split(
        df,
        test_size=(val_ratio + test_ratio),
        stratify=df['domain'],
        random_state=seed,
    )

    # Second split: val vs test (from the temp set)
    relative_test = test_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test,
        stratify=temp_df['domain'],
        random_state=seed,
    )

    return train_df, val_df, test_df


train_df, val_df, test_df = stratified_split(full_triplets, 0.80, 0.10, 0.10, seed=42)

print(f"📊 Split sizes:")
print(f"   Train : {len(train_df):,}  ({len(train_df)/len(full_triplets)*100:.1f}%)")
print(f"   Val   : {len(val_df):,}  ({len(val_df)/len(full_triplets)*100:.1f}%)")
print(f"   Test  : {len(test_df):,}  ({len(test_df)/len(full_triplets)*100:.1f}%)")

📊 Split sizes:
   Train : 9,984  (80.0%)
   Val   : 1,248  (10.0%)
   Test  : 1,248  (10.0%)


In [13]:
# ── Data leakage check ──────────────────────────────────────────────
# Ensure the same positive resume chunk does NOT appear in both train and test
train_positives = set(train_df['positive'].tolist())
test_positives = set(test_df['positive'].tolist())
val_positives = set(val_df['positive'].tolist())

leak_train_test = train_positives & test_positives
leak_train_val = train_positives & val_positives

print(f"\n🔍 Data Leakage Check:")
print(f"   Unique positives in train : {len(train_positives):,}")
print(f"   Unique positives in val   : {len(val_positives):,}")
print(f"   Unique positives in test  : {len(test_positives):,}")
print(f"   Overlap train↔test        : {len(leak_train_test):,}")
print(f"   Overlap train↔val         : {len(leak_train_val):,}")

if leak_train_test:
    pct = len(leak_train_test) / len(test_positives) * 100
    print(f"   ⚠️  {pct:.1f}% of test positives also appear in train.")
    print(f"      This is expected with random sampling from shared pools.")
    print(f"      For production, consider chunk-level deduplication.")
else:
    print(f"   ✅ No leakage detected!")


🔍 Data Leakage Check:
   Unique positives in train : 7,777
   Unique positives in val   : 1,206
   Unique positives in test  : 1,211
   Overlap train↔test        : 466
   Overlap train↔val         : 434
   ⚠️  38.5% of test positives also appear in train.
      This is expected with random sampling from shared pools.
      For production, consider chunk-level deduplication.


In [14]:
# Domain distribution per split
print("📊 Domain distribution per split:\n")
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    counts = split['domain'].value_counts().sort_index()
    total = len(split)
    print(f"── {name} ({total:,} rows) ──")
    for domain, cnt in counts.items():
        bar = '█' * int(cnt / total * 50)
        print(f"   {domain:<15s} {cnt:>5,}  ({cnt/total*100:5.1f}%)  {bar}")
    print()

📊 Domain distribution per split:

── Train (9,984 rows) ──
   Education       1,044  ( 10.5%)  █████
   Finance         1,038  ( 10.4%)  █████
   HR              1,240  ( 12.4%)  ██████
   Healthcare      1,041  ( 10.4%)  █████
   IT              1,052  ( 10.5%)  █████
   Legal           1,032  ( 10.3%)  █████
   Marketing       1,251  ( 12.5%)  ██████
   Operational     1,032  ( 10.3%)  █████
   Sales           1,254  ( 12.6%)  ██████

── Val (1,248 rows) ──
   Education         130  ( 10.4%)  █████
   Finance           130  ( 10.4%)  █████
   HR                155  ( 12.4%)  ██████
   Healthcare        130  ( 10.4%)  █████
   IT                132  ( 10.6%)  █████
   Legal             129  ( 10.3%)  █████
   Marketing         156  ( 12.5%)  ██████
   Operational       129  ( 10.3%)  █████
   Sales             157  ( 12.6%)  ██████

── Test (1,248 rows) ──
   Education         131  ( 10.5%)  █████
   Finance           130  ( 10.4%)  █████
   HR                155  ( 12.4%)  ██████
   

---
## Section 7 · Quality Check

Let's manually inspect some triplets and verify they make sense.

In [15]:
# ── Display 10 random triplets ──────────────────────────────────────
print("🔎 Sample triplets (showing first 200 chars of each):\n")
print("=" * 100)

sample_triplets = train_df.sample(10, random_state=42)
for idx, (_, row) in enumerate(sample_triplets.iterrows(), 1):
    print(f"\n{'─' * 100}")
    print(f"Triplet #{idx}  |  Domain: {row['domain']}  |  "
          f"Neg type: {row.get('neg_type', 'N/A')}  |  Lang: {row.get('lang', 'en')}")
    print(f"{'─' * 100}")
    print(f"  ANCHOR   : {str(row['anchor'])[:200]}...")
    print(f"  POSITIVE : {str(row['positive'])[:200]}...")
    print(f"  NEGATIVE : {str(row['negative'])[:200]}...")
print(f"\n{'=' * 100}")

🔎 Sample triplets (showing first 200 chars of each):


────────────────────────────────────────────────────────────────────────────────────────────────────
Triplet #1  |  Domain: Education  |  Neg type: random  |  Lang: nan
────────────────────────────────────────────────────────────────────────────────────────────────────
  ANCHOR   : Assistant Professor of Psychology Join our team of servant leaders and make a difference in people's lives! Founded in 1894 by The Lutheran Church-Missouri Synod, Concordia University, Nebraska is a p...
  POSITIVE : Education 2012 Bachelors : American Intercontinental University - Business Management - Marketing City , State , US Coursework in Business Management with a concentration in Marketing....
  NEGATIVE : certifications and supervised all security transportation and monitoring needs Managed day to day facility operations and admissions and coordinated daily services including nursing, dining, housekeep...

───────────────────────────────────────

In [16]:
# ── Summary statistics ──────────────────────────────────────────────
print("📈 Triplet Statistics:\n")
print(f"   Total triplets         : {len(full_triplets):,}")
print(f"   English triplets       : {(full_triplets['lang'] == 'en').sum():,}")
print(f"   Indonesian triplets    : {(full_triplets['lang'] == 'id').sum():,}")
print(f"   Random negatives       : {(full_triplets['neg_type'] == 'random').sum():,}")
print(f"   Hard negatives         : {(full_triplets['neg_type'] == 'hard').sum():,}")
print(f"   Unique domains         : {full_triplets['domain'].nunique()}")

print(f"\n📊 Per-domain breakdown:")
domain_stats = full_triplets.groupby('domain').agg(
    total=('anchor', 'count'),
    unique_anchors=('anchor', 'nunique'),
    unique_positives=('positive', 'nunique'),
    unique_negatives=('negative', 'nunique'),
).sort_values('total', ascending=False)
print(domain_stats.to_string())

📈 Triplet Statistics:

   Total triplets         : 12,480
   English triplets       : 0
   Indonesian triplets    : 2,880
   Random negatives       : 11,697
   Hard negatives         : 783
   Unique domains         : 9

📊 Per-domain breakdown:
             total  unique_anchors  unique_positives  unique_negatives
domain                                                                
Sales         1567            1511              1064              1140
Marketing     1564            1430              1008              1134
HR            1550            1506               709              1143
IT            1315            1296              1111               962
Education     1305            1271              1110               969
Healthcare    1301            1293              1079               973
Finance       1298            1265              1157               960
Legal         1290            1233               853               974
Operational   1290            1277            

In [17]:
# ── Text length analysis ────────────────────────────────────────────
print("\n📏 Text length statistics (characters):\n")
for col in ['anchor', 'positive', 'negative']:
    lengths = full_triplets[col].astype(str).str.len()
    print(f"   {col.upper():<10s}  "
          f"mean={lengths.mean():,.0f}  "
          f"median={lengths.median():,.0f}  "
          f"min={lengths.min():,}  "
          f"max={lengths.max():,}")


📏 Text length statistics (characters):

   ANCHOR      mean=1,768  median=1,909  min=156  max=2,403
   POSITIVE    mean=882  median=567  min=80  max=2,410
   NEGATIVE    mean=821  median=497  min=100  max=2,000


In [18]:
# ── Quick semantic sanity check using text length overlap ───────────
# (A rough proxy: same-domain text tends to share more vocabulary)
# For proper semantic check, we'd use the model — but that's Notebook 04.

from difflib import SequenceMatcher

def similarity_score(a, b):
    """Quick string similarity ratio (0–1)."""
    a_short = str(a)[:500]
    b_short = str(b)[:500]
    return SequenceMatcher(None, a_short, b_short).ratio()

print("\n🧪 Semantic Sanity Check (string similarity as rough proxy):")
print("   (Positive should be more similar to anchor than negative)\n")

check_sample = train_df.sample(100, random_state=42)
pos_sims = [similarity_score(r['anchor'], r['positive']) for _, r in check_sample.iterrows()]
neg_sims = [similarity_score(r['anchor'], r['negative']) for _, r in check_sample.iterrows()]

pos_better = sum(1 for p, n in zip(pos_sims, neg_sims) if p > n)
print(f"   Avg similarity (anchor↔positive) : {np.mean(pos_sims):.4f}")
print(f"   Avg similarity (anchor↔negative) : {np.mean(neg_sims):.4f}")
print(f"   Positive > Negative in {pos_better}/100 cases ({pos_better}%)")

if np.mean(pos_sims) > np.mean(neg_sims):
    print("   ✅ Positive is on average MORE similar to anchor — good signal!")
else:
    print("   ⚠️  Similarity proxy inconclusive — this is normal for short text.")
    print("      The model will learn deeper semantic features during training.")


🧪 Semantic Sanity Check (string similarity as rough proxy):
   (Positive should be more similar to anchor than negative)

   Avg similarity (anchor↔positive) : 0.0518
   Avg similarity (anchor↔negative) : 0.0500
   Positive > Negative in 55/100 cases (55%)
   ✅ Positive is on average MORE similar to anchor — good signal!


---
## Section 8 · Save Triplets to Google Drive

In [19]:
# Select columns to save (drop internal metadata)
save_cols = ['anchor', 'positive', 'negative', 'domain']

train_path = os.path.join(TRIPLET_DIR, 'train.csv')
val_path = os.path.join(TRIPLET_DIR, 'val.csv')
test_path = os.path.join(TRIPLET_DIR, 'test.csv')

train_df[save_cols].to_csv(train_path, index=False)
val_df[save_cols].to_csv(val_path, index=False)
test_df[save_cols].to_csv(test_path, index=False)

print("💾 Saved triplet splits to Google Drive:")
print(f"   {train_path}  ({len(train_df):,} rows)")
print(f"   {val_path}  ({len(val_df):,} rows)")
print(f"   {test_path}  ({len(test_df):,} rows)")

# Also save the full combined dataset (with metadata) for reference
full_path = os.path.join(TRIPLET_DIR, 'triplets_full.csv')
full_triplets.to_csv(full_path, index=False)
print(f"   {full_path}  ({len(full_triplets):,} rows, includes lang/neg_type metadata)")

💾 Saved triplet splits to Google Drive:
   /content/drive/MyDrive/CVPRO/data/triplets/train.csv  (9,984 rows)
   /content/drive/MyDrive/CVPRO/data/triplets/val.csv  (1,248 rows)
   /content/drive/MyDrive/CVPRO/data/triplets/test.csv  (1,248 rows)
   /content/drive/MyDrive/CVPRO/data/triplets/triplets_full.csv  (12,480 rows, includes lang/neg_type metadata)


In [20]:
# ── Final verification: reload and check ────────────────────────────
train_reload = pd.read_csv(train_path)
val_reload = pd.read_csv(val_path)
test_reload = pd.read_csv(test_path)

print("\n✅ Reload verification:")
print(f"   train.csv : {train_reload.shape}  columns={list(train_reload.columns)}")
print(f"   val.csv   : {val_reload.shape}  columns={list(val_reload.columns)}")
print(f"   test.csv  : {test_reload.shape}  columns={list(test_reload.columns)}")
print(f"\n   Total     : {len(train_reload) + len(val_reload) + len(test_reload):,} triplets")


✅ Reload verification:
   train.csv : (9984, 4)  columns=['anchor', 'positive', 'negative', 'domain']
   val.csv   : (1248, 4)  columns=['anchor', 'positive', 'negative', 'domain']
   test.csv  : (1248, 4)  columns=['anchor', 'positive', 'negative', 'domain']

   Total     : 12,480 triplets


In [21]:
# ── Summary ─────────────────────────────────────────────────────────
print("=" * 60)
print("  📐 TRIPLET GENERATION COMPLETE")
print("=" * 60)
print(f"""
  Base triplets (EN, random neg) : {(full_triplets['neg_type'] == 'random').sum():,}
  Hard negatives (EN)            : {(full_triplets['neg_type'] == 'hard').sum():,}
  Indonesian augmented           : {(full_triplets['lang'] == 'id').sum():,}
  ─────────────────────────────────
  Total triplets                 : {len(full_triplets):,}

  Train split                    : {len(train_df):,}
  Val split                      : {len(val_df):,}
  Test split                     : {len(test_df):,}

  Saved to: {TRIPLET_DIR}

  Next step → Notebook 04: Fine-tune paraphrase-multilingual-MiniLM-L12-v2
""")
print("=" * 60)

  📐 TRIPLET GENERATION COMPLETE

  Base triplets (EN, random neg) : 11,697
  Hard negatives (EN)            : 783
  Indonesian augmented           : 2,880
  ─────────────────────────────────
  Total triplets                 : 12,480

  Train split                    : 9,984
  Val split                      : 1,248
  Test split                     : 1,248

  Saved to: /content/drive/MyDrive/CVPRO/data/triplets

  Next step → Notebook 04: Fine-tune paraphrase-multilingual-MiniLM-L12-v2

